# Детекция скраперов по cookie_id

Для каждой пары «кука и сутки» я возвращаю число от 0 до 1: насколько вероятно, что за кукой
стоит автоматический сборщик данных.

Метрика задачи: precision при recall не ниже 0.70. Я обязан поймать минимум 70% ботов и среди
всех способов это сделать выбрать наименее ложно-тревожный. Порог подбирает проверяющая система,
поэтому я отдаю только числа, а калибровать их не нужно: метрике важен лишь порядок кук.

**Результат на тесте: P@R≥0.70 = 0.883.** Константный ответ там даёт 0.094.

Собственная оценка по кросс-валидации была 0.874. Тест оказался чуть выше, и это ожидаемо:
финальная модель учится на всех размеченных строках, а каждая модель внутри кросс-валидации
видит четыре пятых из них. Плюс в тесте ботов относительно больше, а чем их больше, тем легче
держать precision при той же полноте.

Ноутбук самодостаточен: весь код внутри, включая метрику, внешних модулей проекта не требуется.
Прогон около минуты. Здесь только итоговое решение, а весь ход работы, проверки и отклонённые
идеи лежат в папке `theories`.

In [8]:
import re
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

EVENT_NAMES = ["search_results_view", "item_view", "photo_swipe", "seller_page_view",
               "contact_phone_show", "contact_chat_open", "contact_message_sent",
               "favorite_add", "login"]
CLIENTS = ["Scrapy", "curl", "node-fetch", "urllib3", "requests", "Go-http-client"]
UA_FAMILIES = ["chrome", "firefox", "safari", "yabrowser", "avito_app", "client"]
UA_OSES = ["windows", "macos", "linux_x11", "android", "ios"]
PLATFORMS = ["desktop", "web", "android", "ios"]
TE_COLS = ["item_te_mean", "item_te_max", "item_known_frac"]


# Метрика кейса. Код скопирован из metric.py организаторов без единого изменения, чтобы мои
# локальные числа считались ровно тем же, чем считает проверяющая система.

def pr_curve(y_true, score):
    """(precision, recall) в точках на границах групп одинакового score."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if y_true.shape != score.shape:
        raise ValueError("y_true и score разной длины")

    n_pos = int(y_true.sum())
    if n_pos == 0:
        return np.array([]), np.array([])

    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]

    tp = np.cumsum(y)
    k = np.arange(1, len(y) + 1)
    ends = np.r_[s[1:] != s[:-1], True]      # последняя строка каждой группы равных score
    return tp[ends] / k[ends], tp[ends] / n_pos


def precision_at_recall(y_true, score, recall=0.70):
    """Максимальный precision среди порогов с recall >= `recall`."""
    prec, rec = pr_curve(y_true, score)
    if len(prec) == 0:
        return float("nan")
    ok = rec >= recall
    return float(prec[ok].max()) if ok.any() else 0.0


# самопроверка копии на примерах из описания метрики
assert precision_at_recall([1, 0, 1, 1, 1], [5, 4, 3, 2, 1]) == 0.8
assert precision_at_recall([1, 1, 0, 0], [0.5] * 4) == 0.5    # равные score идут одной группой
assert precision_at_recall([0, 0, 1, 1], [0.5] * 4) == 0.5    # порядок строк не влияет

In [9]:
def clip_to_window(events_raw, meta):
    """Единственная дверь к событиям: только то, что попало внутрь окна строки.

    Всё за правой границей окна лежит в будущем относительно момента решения. Assert ловит captcha_shown,
    которая целиком лежит после окна.
    """
    ev = events_raw.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]],
                          on="cookie_id", how="inner")
    keep = (ev["event_ts"] >= ev["window_start_ts"]) & (ev["event_ts"] < ev["window_end_ts"])
    ev = ev.loc[keep].sort_values(["cookie_id", "event_ts"], kind="mergesort").reset_index(drop=True)
    ev["platform_norm"] = ev["platform"].astype("string").str.lower().replace({"iphone": "ios"})
    assert (ev["event_name"] == "captcha_shown").sum() == 0
    return ev


def entropy(s):
    p = s.value_counts(normalize=True)
    return float(-(p * np.log(p)).sum()) if len(p) else np.nan


def ua_parse(s):
    fam = "client" if any(c in s for c in CLIENTS) else next(
        (n for k, n in [("okhttp", "avito_app"), ("YaBrowser", "yabrowser"), ("Firefox", "firefox"),
                        ("Chrome", "chrome"), ("Safari", "safari")] if k in s), "other")
    # порядок важен: в строках айфона есть "like Mac OS X", поэтому мобильные проверяем первыми
    os_ = next((n for k, n in [("Windows NT", "windows"), ("Android", "android"),
                               ("iPhone", "ios"), ("iPad", "ios"),
                               ("Macintosh", "macos"), ("Mac OS X", "macos"),
                               ("X11", "linux_x11")] if k in s), "other")
    ver = np.nan
    for pat in [r"Chrome/(\d+)", r"Firefox/(\d+)", r"Version/(\d+)", r"Avito/(\d+)", r"Scrapy/(\d+)",
                r"curl/(\d+)", r"node-fetch/(\d+)", r"urllib3/(\d+)", r"requests/(\d+)", r"Go-http-client/(\d+)"]:
        m = re.search(pat, s)
        if m:
            ver = float(m.group(1))
            break
    return fam, os_, ver


def population_stats(events, train, test):
    """Популярность объявления и частота UA накопительно: только по прошлым дням."""
    ev = pd.concat([clip_to_window(events, train), clip_to_window(events, test)], ignore_index=True)
    pr = ev.dropna(subset=["item_id"])[["item_id", "cookie_id", "window_start_ts"]].drop_duplicates()
    day = pr.groupby(["item_id", "window_start_ts"]).size().unstack(fill_value=0).sort_index(axis=1)
    item_pop = day.cumsum(axis=1).shift(1, axis=1).fillna(0).stack()

    up = ev[["user_agent", "cookie_id", "window_start_ts"]].drop_duplicates()
    ud = up.groupby(["user_agent", "window_start_ts"]).size().unstack(fill_value=0).sort_index(axis=1)
    ub = ud.cumsum(axis=1).shift(1, axis=1).fillna(0)
    ua_freq = (ub / ub.sum(axis=0).replace(0, np.nan)).stack()

    ua = pd.Series(events["user_agent"].unique())
    parsed = [ua_parse(s) for s in ua]
    info = pd.DataFrame({"user_agent": ua.values,
                         "ua_family": [p[0] for p in parsed],
                         "ua_os": [p[1] for p in parsed],
                         "ua_ver": [p[2] for p in parsed]}).set_index("user_agent")
    info["ua_is_client"] = (info["ua_family"] == "client").astype(float)
    return {"item_pop": item_pop, "ua_freq": ua_freq, "ua_info": info}

In [10]:
def build_features(meta, events_raw, pop):
    """86 признаков поведения куки. Одна функция на train и test, чтобы наборы не разъехались."""
    ev = clip_to_window(events_raw, meta)
    idx = pd.Index(meta["cookie_id"].values, name="cookie_id")
    F = pd.DataFrame(index=idx)
    g = ev.groupby("cookie_id", sort=False)
    win_h = (meta["window_end_ts"] - meta["window_start_ts"]).dt.total_seconds().values / 3600.0
    cookie_day = pd.Series(meta["window_start_ts"].values, index=meta["cookie_id"].values)

    def put(name, val):
        F[name] = val.reindex(idx) if isinstance(val, pd.Series) else val

    # объём: сколько событий и на сколько разных объявлений их хватило
    put("n_events", g.size())
    F["n_events"] = F["n_events"].fillna(0)
    put("n_items_uniq", g["item_id"].nunique())
    put("n_cat_uniq", g["item_category"].nunique())
    put("n_loc_uniq", g["item_location"].nunique())
    put("n_query_uniq", g["search_query"].nunique())
    put("events_per_hour", F["n_events"].values / win_h)
    cnt = pd.crosstab(ev["cookie_id"], ev["event_name"]).reindex(columns=EVENT_NAMES, fill_value=0)
    cnt = cnt.reindex(idx).fillna(0.0)
    for c in EVENT_NAMES:
        put("cnt_" + c, cnt[c])

    ev = ev.copy()
    ev["dt"] = g["event_ts"].diff().dt.total_seconds()
    gd = ev.dropna(subset=["dt"]).groupby("cookie_id")["dt"]
    put("dt_median", gd.median()); put("dt_min", gd.min()); put("dt_std", gd.std())
    put("dt_frac_lt_1s", gd.apply(lambda s: (s < 1).mean()))
    put("span_seconds", (g["event_ts"].max() - g["event_ts"].min()).dt.total_seconds())
    put("n_active_hours", ev.assign(h=ev["event_ts"].dt.hour).groupby("cookie_id")["h"].nunique())

    views = F["cnt_item_view"].clip(lower=1)
    contacts = F["cnt_contact_phone_show"] + F["cnt_contact_chat_open"] + F["cnt_contact_message_sent"]
    put("photo_per_view", F["cnt_photo_swipe"] / views)
    put("contact_per_view", contacts / views)
    put("view_per_search", F["cnt_item_view"] / F["cnt_search_results_view"].clip(lower=1))
    put("max_search_page", g["search_page"].max())
    put("mean_search_page", g["search_page"].mean())
    put("ptr_frac", g["pointer_x"].apply(lambda s: s.notna().mean()))
    put("ptr_x_nunique", g["pointer_x"].nunique())
    xy = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()
    xy["_xy"] = xy["pointer_x"].astype(int).astype(str) + "_" + xy["pointer_y"].astype(int).astype(str)
    put("ptr_xy_nunique", xy.groupby("cookie_id")["_xy"].nunique())
    F["ptr_xy_nunique"] = F["ptr_xy_nunique"].fillna(0.0)
    put("n_platform_norm", g["platform_norm"].nunique())
    put("n_user_agent", g["user_agent"].nunique())
    put("cookie_age_days",
        (meta["window_end_ts"] - meta["cookie_created_at"]).dt.total_seconds().values / 86400.0)
    put("window_dow", meta["window_start_ts"].dt.dayofweek.values.astype(float))

    # разброс по каталогу: сидит в одной теме или ходит по всему сайту
    evc = ev.dropna(subset=["item_category"])
    put("cat_entropy", evc.groupby("cookie_id")["item_category"].apply(entropy))
    put("cat_top_frac", evc.groupby("cookie_id")["item_category"].apply(
        lambda s: s.value_counts(normalize=True).max()))
    evc = evc.copy(); evc["prev"] = evc.groupby("cookie_id")["item_category"].shift()
    put("cat_switch_frac", evc.assign(
        sw=((evc["prev"].notna()) & (evc["prev"] != evc["item_category"])).astype(float)
    ).groupby("cookie_id")["sw"].mean())
    evl = ev.dropna(subset=["item_location"])
    put("loc_entropy", evl.groupby("cookie_id")["item_location"].apply(entropy))
    put("loc_top_frac", evl.groupby("cookie_id")["item_location"].apply(
        lambda s: s.value_counts(normalize=True).max()))
    evi = ev.dropna(subset=["item_id"]); gi = evi.groupby("cookie_id")
    put("item_repeat_frac", 1 - gi["item_id"].nunique() / gi.size())
    key = pd.MultiIndex.from_arrays([evi["item_id"].values, evi["window_start_ts"].values])
    put("item_pop_mean", evi.assign(p=pop["item_pop"].reindex(key).values)
        .groupby("cookie_id")["p"].mean())
    stt = pd.crosstab(ev["cookie_id"], ev["seller_type"], normalize="index")
    put("seller_frac_private", stt["private"] if "private" in stt else pd.Series(0.0, index=stt.index))
    put("seller_known_frac", g["seller_type"].apply(lambda s: s.notna().mean()))

    # ритм: длина пауз между действиями и насколько они ровные
    put("dt_mean", gd.mean()); put("dt_max", gd.max())
    put("dt_p10", gd.quantile(0.10)); put("dt_p90", gd.quantile(0.90))
    put("dt_frac_lt_5s", gd.apply(lambda s: (s < 5).mean()))
    F["dt_cv"] = F["dt_std"] / F["dt_mean"].clip(lower=1e-9)
    ev["new_sess"] = ev["dt"].isna() | (ev["dt"] > 1800)
    ev["sess_id"] = ev.groupby("cookie_id")["new_sess"].cumsum()
    sess = ev.groupby(["cookie_id", "sess_id"]).size().rename("n").reset_index()
    put("n_sessions", sess.groupby("cookie_id")["sess_id"].max())
    put("sess_events_mean", sess.groupby("cookie_id")["n"].mean())
    put("sess_events_max", sess.groupby("cookie_id")["n"].max())
    put("hour_entropy", ev.assign(h=ev["event_ts"].dt.hour).groupby("cookie_id")["h"].apply(entropy))
    put("night_frac", ev.assign(n=(ev["event_ts"].dt.hour < 6).astype(float))
        .groupby("cookie_id")["n"].mean())

    # поиск: как глубоко листает выдачу и повторяет ли один запрос
    srch = ev[ev["event_name"] == "search_results_view"].copy(); gs = srch.groupby("cookie_id")
    put("page_gt5_frac", gs["search_page"].apply(lambda s: (s > 5).mean()))
    put("page_gt10_frac", gs["search_page"].apply(lambda s: (s > 10).mean()))
    put("page_std", gs["search_page"].std())
    srch["prev_page"] = gs["search_page"].shift()
    put("page_step_plus1_frac", srch.assign(
        st=((srch["search_page"] - srch["prev_page"]) == 1).astype(float)
    ).groupby("cookie_id")["st"].mean())
    put("q_repeat_frac", 1 - gs["search_query"].nunique() / gs.size().clip(lower=1))
    put("q_len_mean", gs["search_query"].apply(lambda s: s.dropna().str.len().mean()))

    # курсор: есть ли координаты вообще и как он по экрану ходит
    pxy = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()
    pxy["dist"] = np.sqrt(pxy.groupby("cookie_id")["pointer_x"].diff() ** 2
                          + pxy.groupby("cookie_id")["pointer_y"].diff() ** 2)
    pxy["_xy"] = pxy["pointer_x"].astype(int).astype(str) + "_" + pxy["pointer_y"].astype(int).astype(str)
    gp = pxy.groupby("cookie_id")
    put("ptr_x_std", gp["pointer_x"].std()); put("ptr_y_std", gp["pointer_y"].std())
    put("ptr_dist_mean", gp["dist"].mean()); put("ptr_dist_std", gp["dist"].std())
    put("ptr_repeat_frac", 1 - gp["_xy"].nunique() / gp.size())

    # устройство: браузер, ОС, платформа и редкость такого user-agent
    ua_ck = ev.groupby("cookie_id")["user_agent"].agg(lambda s: s.mode().iat[0])
    uj = pop["ua_info"].reindex(ua_ck.values)
    put("ua_is_client", pd.Series(uj["ua_is_client"].values, index=ua_ck.index))
    put("ua_ver", pd.Series(uj["ua_ver"].values, index=ua_ck.index))
    uk = pd.MultiIndex.from_arrays([ua_ck.values, cookie_day.reindex(ua_ck.index).values])
    put("ua_freq", pd.Series(pop["ua_freq"].reindex(uk).values, index=ua_ck.index))
    for fam in UA_FAMILIES:
        put("ua_fam_" + fam, pd.Series((uj["ua_family"].values == fam).astype(float), index=ua_ck.index))
    for os_ in UA_OSES:
        put("ua_os_" + os_, pd.Series((uj["ua_os"].values == os_).astype(float), index=ua_ck.index))
    pl = pd.crosstab(ev["cookie_id"], ev["platform_norm"], normalize="index")
    for p in PLATFORMS:
        put("plat_frac_" + p, pl[p] if p in pl else pd.Series(0.0, index=pl.index))
    ev["prev_plat"] = ev.groupby("cookie_id")["platform_norm"].shift()
    put("plat_switch_frac", ev.assign(
        sw=((ev["prev_plat"].notna()) & (ev["prev_plat"] != ev["platform_norm"])).astype(float)
    ).groupby("cookie_id")["sw"].mean())

    zero = ["n_events", "events_per_hour", "ptr_xy_nunique"] + ["cnt_" + c for c in EVENT_NAMES]
    F[zero] = F[zero].fillna(0.0)
    return F.reset_index(drop=True)

In [11]:
def cov_index(events, train, test):
    """Кто с кем пересекался по объявлениям, только по прошлым дням.

    Скраперы обходят каталог по спискам, и списки у разных сборщиков пересекаются. Двум живым
    людям совпасть по нескольким объявлениям почти нереально.
    """
    ev = pd.concat([clip_to_window(events, train), clip_to_window(events, test)])
    pr = ev.dropna(subset=["item_id"])[["cookie_id", "item_id", "window_start_ts"]].drop_duplicates()
    part = defaultdict(Counter)
    for _, grp in pr.groupby("item_id"):
        ck, dy = grp["cookie_id"].values, grp["window_start_ts"].values
        if len(ck) < 2 or len(ck) > 60:      # слишком популярные связывают всех со всеми
            continue
        for i in range(len(ck)):
            for j in range(len(ck)):
                if i != j and dy[j] < dy[i]:
                    part[ck[i]][ck[j]] += 1
    return part


def cov_features(meta, cov):
    idx = pd.Index(meta["cookie_id"].values)
    f = lambda d: pd.Series(d).reindex(idx).fillna(0).values
    return pd.DataFrame({
        "cov_partners": f({k: len(v) for k, v in cov.items()}),
        "cov_max_shared": f({k: max(v.values()) for k, v in cov.items()}),
        "cov_strong": f({k: sum(1 for x in v.values() if x >= 2) for k, v in cov.items()})})


def item_pairs(meta, events):
    ev = clip_to_window(events, meta).dropna(subset=["item_id"])
    p = ev[["cookie_id", "item_id"]].drop_duplicates()
    pos = pd.Series(np.arange(len(meta)), index=meta["cookie_id"].values)
    return p.assign(row=pos.reindex(p["cookie_id"]).values)[["row", "item_id"]]


def encode_items(fit_pairs, fit_y, apply_pairs, n_apply, k=10.0, drop_own=True):
    """Репутация объявления: доля ботов среди его зрителей, со сглаживанием.

    Статистика только по fit_pairs. Для обучающих строк вычитается собственный вклад, иначе
    признак предсказывал бы метку, из которой сам сделан.
    """
    p = float(np.mean(np.asarray(fit_y)[np.unique(fit_pairs["row"].values)]))
    fp = fit_pairs.assign(y=np.asarray(fit_y)[fit_pairs["row"].values])
    st = fp.groupby("item_id")["y"].agg(["sum", "size"])
    s = apply_pairs["item_id"].map(st["sum"]).fillna(0.0).values
    n = apply_pairs["item_id"].map(st["size"]).fillna(0.0).values
    if drop_own:
        s = s - np.asarray(fit_y)[apply_pairs["row"].values].astype(float)
        n = n - 1.0
    d = pd.DataFrame({"row": apply_pairs["row"].values, "enc": (s + k * p) / (n + k),
                      "known": (n > 0).astype(float)})
    g = d.groupby("row")
    out = np.full((n_apply, 3), np.nan)
    out[g["enc"].mean().index.values, 0] = g["enc"].mean().values
    out[g["enc"].max().index.values, 1] = g["enc"].max().values
    out[g["known"].mean().index.values, 2] = g["known"].mean().values
    return out

In [12]:
dates = ["cookie_created_at", "window_start_ts", "window_end_ts"]
train = pd.read_csv("data/train.csv", parse_dates=dates)
test = pd.read_csv("data/test.csv", parse_dates=dates)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])
ytr = train["target"].values.astype(int)

pop = population_stats(events, train, test)
Xtr, Xte = build_features(train, events, pop), build_features(test, events, pop)
cov = cov_index(events, train, test)
Xtr = pd.concat([Xtr, cov_features(train, cov)], axis=1)
Xte = pd.concat([Xte, cov_features(test, cov)], axis=1)
FEATURES = list(Xtr.columns)
P_tr, P_te = item_pairs(train, events), item_pairs(test, events)

print("train", train.shape, "| test", test.shape, "| ботов", ytr.sum(), f"({ytr.mean():.1%})")
print("признаков:", len(FEATURES), "плюс 3 кодировки, которые считаются внутри фолдов")

train (11091, 5) | test (4909, 4) | ботов 899 (8.1%)
признаков: 86 плюс 3 кодировки, которые считаются внутри фолдов


## Что я пробовал

Признак имеет смысл, если отвечает на вопрос, в котором человек и сборщик данных расходятся.
Таких вопросов у меня набралось восемь.

Шесть про саму куку. Объём: человек за день выдыхается на паре десятков карточек, сборщику
уставать нечем. Разброс по каталогу: живой ищет конкретную вещь и обычно в своём городе,
обходчик идёт вширь ровным слоем. Ритм: у человека паузы рваные, он отвлекается и уходит, у
автомата они ровные, даже если он специально замедлен. Поиск: человек смотрит первые страницы и
возвращается назад, сборщик листает подряд до конца. Курсор: у скрипта на requests координат нет
вовсе, потому что нет и курсора. Устройство: браузер, ОС и то, насколько редок такой user-agent.

Два оставшихся вопроса про связи между куками, и они дали больше всего. Сколько других кук
смотрели те же объявления, и какова репутация этих объявлений, то есть доля ботов среди тех, кто
их уже смотрел. Одиночное поведение я к тому моменту выжал, а того, что разные куки ходят по
одним и тем же спискам, признаки по одной куке не видят в принципе.

Три места, где легко получить красивое число и обмануть себя.

Событие `captcha_shown` встречается у половины ботов и почти не встречается у людей, но лежит
целиком после правой границы окна. Это реакция площадки на куку, уже признанную подозрительной,
то есть ответ, а не улика. Единственная дверь к событиям в ноутбуке это `clip_to_window`, и в ней
стоит assert, который не даст этой строке просочиться.

Популярность объявления, посчитанная сразу по всем дням, заметно поднимает метрику. Это знание
будущего: в момент решения ещё неизвестно, сколько человек посмотрят объявление завтра. Честная
накопительная версия знает только прошлое и даёт скромнее.

Кодировка по объявлениям без вычитания собственного вклада не завышает результат, а разрушает
его. Признак начинает предсказывать метку, из которой сам сделан, модель опирается на него
целиком, а на проверке такой опоры нет.

Что я проверил и не взял:

- порядок переходов между типами событий: прирост на уровне шума;
- компоненты PCA: на уровне шума, помогают одной модели и вредят другой;
- признаки из разбора кода настоящих скраперов: поодиночке сильные, сверх текущих не дают ничего;
- рассогласование user-agent и платформы: почти копия уже имеющегося признака;
- очереди одинаковых действий и треугольники в графе: знак скачет между протоколами;
- веса классов и ранжирующая целевая функция: двигают калибровку, а не порядок кук, а метрике
  важен только порядок;
- кодировка категорий и городов метками: значений мало и они между собой не различаются, признак
  начинает нести артефакт вместо сигнала и ухудшает предсказание;
- расширения графа связей, то есть вес по редкости объявления, второй порядок, метки соседей,
  совпадение порядка обхода: всё на уровне шума;
- псевдоразметка, синтетические боты, усреднение по нескольким зёрнам, смеси моделей: не
  улучшают;
- перебор гиперпараметров: полтораста конфигураций по пятнадцати параметрам, лучшая почти
  совпала с текущими настройками, а её перевес исчез на разбиениях, которых перебор не видел;
- ранняя остановка обучения, в том числе по нашей же метрике: обрывает рост леса примерно на сотне
  деревьев вместо пятисот и недоучивает, вдобавок отнимает часть фолда под валидацию.

Из двадцати двух проверенных идей приняты две, и обе про связи между куками.

In [6]:
def make_xgb():
    return XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
                         colsample_bytree=0.8, tree_method="hist", eval_metric="logloss",
                         random_state=SEED, verbosity=0, n_jobs=-1)


def oof(folds):
    """Предсказания вне обучения. Кодировка пересчитывается внутри каждого фолда."""
    p = np.full(len(ytr), np.nan)
    for a, b in folds:
        fp, vp = P_tr[P_tr.row.isin(set(a))], P_tr[P_tr.row.isin(set(b))]
        A, B = Xtr.iloc[a][FEATURES].copy(), Xtr.iloc[b][FEATURES].copy()
        ea = encode_items(fp, ytr, fp, len(ytr), drop_own=True)
        eb = encode_items(fp, ytr, vp, len(ytr), drop_own=False)
        for i, c in enumerate(TE_COLS):
            A[c], B[c] = ea[a, i], eb[b, i]
        m = make_xgb()
        m.fit(A, ytr[a])
        p[b] = m.predict_proba(B)[:, 1]
    return p


p_oof = oof(list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xtr, ytr)))

prec, rec = pr_curve(ytr, p_oof)
srt = np.sort(p_oof)[::-1]
thr = srt[np.r_[srt[1:] != srt[:-1], True]]
ok = rec >= 0.70
i = np.where(ok)[0][int(np.argmax(prec[ok]))]
tn, fp_, fn, tp = confusion_matrix(ytr, p_oof >= thr[i]).ravel()

print(f"P@R>=0.70   {prec[i]:.4f}       ROC-AUC  {roc_auc_score(ytr, p_oof):.4f}")
print(f"recall      {rec[i]:.4f}       PR-AUC   {average_precision_score(ytr, p_oof):.4f}")
print()
print(f"помечено кук      {tp + fp_} из {len(ytr)}   (боты {tp}, ложные тревоги {fp_})")
print(f"пропущено ботов   {fn} из {ytr.sum()}")
print(f"задето людей      {fp_ / (fp_ + tn):.2%}")

P@R>=0.70   0.8736       ROC-AUC  0.9421
recall      0.7075       PR-AUC   0.8245

помечено кук      728 из 11091   (боты 636, ложные тревоги 92)
пропущено ботов   263 из 899
задето людей      0.90%


In [7]:
Xtr_fin, Xte_fin = Xtr[FEATURES].copy(), Xte[FEATURES].copy()
e_tr = encode_items(P_tr, ytr, P_tr, len(ytr), drop_own=True)
e_te = encode_items(P_tr, ytr, P_te, len(test), drop_own=False)
for i, c in enumerate(TE_COLS):
    Xtr_fin[c], Xte_fin[c] = e_tr[:, i], e_te[:, i]

model = make_xgb()
model.fit(Xtr_fin, ytr)
pred = model.predict_proba(Xte_fin)[:, 1]

sample = pd.read_csv("data/sample_submission.csv")
sub = (pd.DataFrame({"cookie_id": test.cookie_id.values, "score": pred})
       .set_index("cookie_id").reindex(sample.cookie_id).reset_index())
assert len(sub) == len(test) == 4909
assert list(sub.columns) == ["cookie_id", "score"]
assert sub.cookie_id.is_unique and sub.notna().all().all()
assert sub.cookie_id.tolist() == sample.cookie_id.tolist()
assert sub.score.between(0, 1).all()
sub.to_csv("submission.csv", index=False)

print("обучено на", len(Xtr), "строках,", len(FEATURES) + len(TE_COLS), "признаков")
print("submission.csv записан:", sub.shape, "| проверки формата пройдены")
display(sub.head())

обучено на 11091 строках, 89 признаков
submission.csv записан: (4909, 2) | проверки формата пройдены


,cookie_id,score
0,ck_99a4e5ef02d89493,0.004197
1,ck_54d463f7fd89ec3d,0.005949
2,ck_0fbbc7a6af300368,0.000626
3,ck_8fd4937eadd64fb6,0.000335
4,ck_dbb4bd4eda97255d,0.000716


## Почему я остановился здесь

Лидерборда у задачи нет, поэтому единственный ориентир это собственная валидация, а её легко
переоптимизировать. Шум метрики я измерил парным bootstrap на одних и тех же перевыборках.
Улучшение меньше этого шума неотличимо от того, какие именно куки попали в оценку, а если
перебрать десяток идей и взять лучшую, она окажется завышена просто по арифметике выбора.

Поэтому правило приёмки я записал до экспериментов, а не после: идея принимается, только если
bootstrap уверенно за неё, знак сохраняется на всех повторах разбиения и хронологический протокол
тоже не падает. Подавляющее большинство проверенных идей это правило не прошло.

Дальше упирается не модель, а данные. Я разобрал ботов, которых решение не ловит, и сравнил их с
людьми по всем признакам модели плюс несколько десятков величин, посчитанных специально для этой
проверки. Ни одна из них не отделяет непойманных ботов от людей заметно лучше монетки, тогда как
пойманных отделяют десятки признаков.

Причина в объёме следа. У непойманных ботов событий за сутки примерно столько же, сколько у
обычного человека. Бот, сделавший дюжину действий, не успевает проявить ни ритма, ни глубины
выдачи, ни широты обхода. Связи тоже не выручают: чтобы делить объявление с кем-то, надо сначала
посмотреть объявления, поэтому у кук с коротким следом граф так же пуст, как и поведение.

Отдельно про мышь, раз эта группа оказалась одной из сильнейших. Координаты в данных существуют
только на настольных платформах, на мобильных их нет ни у кого. Значит признак работает не как
«бот прячет мышь», а как пара условий «пришёл с десктопа и мышью не двигает». Явный
признак-взаимодействие я делал и отклонил: дерево строит эту пару разбиений само.

Помогло бы только то, чего в выгрузке нет: история куки за прошлые дни, отпечаток устройства,
окно длиннее одних суток.